In [13]:
import pandas as pd
import numpy as np

df = pd.read_parquet("History.parquet")
cards = pd.read_pickle("cards.pkl")

final_games = (
    df
    .loc[df["runde"] == 4]
    .drop_duplicates(subset="game_id")
    .copy()
)

n_games = len(final_games)
total_sp = final_games["sp"].sum()

print(f"Games analysed: {n_games:,}")

Games analysed: 10,000


# Analyze cards played
One row per play

In [14]:
plays_df = (
    final_games[
        ["game_id", "sp", "played_cards_log"]
    ]
    .explode("played_cards_log", ignore_index=True)
    .dropna(subset=["played_cards_log"])
)

plays_df["card_id"] = plays_df["played_cards_log"].apply(
    lambda entry: entry["card_id"]
)

plays_df["played_round"] = plays_df["played_cards_log"].apply(
    lambda entry: entry["runde"]
)

plays_df = (
    plays_df
    .drop(columns="played_cards_log")
    .merge(
        cards[
            ["id", "Name", "Slot/Stapel", "Kosten"]
        ].drop_duplicates("id"),
        left_on="card_id",
        right_on="id",
        how="left",
        validate="many_to_one",
    )
    .drop(columns="id")
)
#3. Ensure each card counts only once per game
game_cards = (
    plays_df[
        [
            "game_id",
            "card_id",
            "Name",
            "Slot/Stapel",
            "Kosten",
            "sp",
            "played_round",
        ]
    ]
    .drop_duplicates(["game_id", "card_id"])
)

print(f"Card Plays analysed: {len(game_cards):,}")

Card Plays analysed: 135,934


# Compare final SP with and without each card

In [15]:

card_results = (
    game_cards
    .groupby(
        ["card_id", "Name", "Slot/Stapel", "Kosten"],
        dropna=False,
    )
    .agg(
        games_with_card=("game_id", "nunique"),
        sp_sum_with_card=("sp", "sum"),
        mean_sp_with_card=("sp", "mean"),
        mean_played_round=("played_round", "mean"),
    )
    .reset_index()
)

card_results["games_without_card"] = (
    n_games - card_results["games_with_card"]
)

card_results["mean_sp_without_card"] = np.where(
    card_results["games_without_card"] > 0,
    (
        total_sp - card_results["sp_sum_with_card"]
    ) / card_results["games_without_card"],
    np.nan,
)

card_results["sp_uplift"] = (
    card_results["mean_sp_with_card"]
    - card_results["mean_sp_without_card"]
)

card_results["play_rate"] = (
    card_results["games_with_card"] / n_games
)

In [ ]:
minimum_games = 100

reliable_cards = (
    card_results
    .loc[card_results["games_with_card"] >= minimum_games]
    .sort_values("sp_uplift", ascending=False)
)

reliable_cards[
    [
        "Name",
        "Slot/Stapel",
        "Kosten",
        "games_with_card",
        "play_rate",
        "mean_played_round",
        "mean_sp_with_card",
        "mean_sp_without_card",
        "sp_uplift",
    ]
].head(30)

KeyError: "['best_percent_games', 'best_percent_rate_when_played'] not in index"